# 特征编码器使用演示

本 Notebook 演示 hscredit 中各类特征编码器的使用方法。

支持的编码器：
- **WOEEncoder**: 证据权重编码（评分卡专用）
- **TargetEncoder**: 目标编码
- **CountEncoder**: 计数编码
- **OneHotEncoder**: 独热编码
- **OrdinalEncoder**: 序数编码
- **QuantileEncoder**: 分位数编码
- **CatBoostEncoder**: CatBoost编码

In [1]:
import sys
sys.path.insert(0, '/Users/xiaoxi/CodeBuddy/hscredit/hscredit')

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

from hscredit.core.encoders import (
    WOEEncoder, TargetEncoder, CountEncoder,
    OneHotEncoder, OrdinalEncoder, QuantileEncoder,
    CatBoostEncoder
)

np.random.seed(42)

## 1. 创建测试数据

In [2]:
# 创建测试数据
n = 2000

# 类别特征
city = np.random.choice(['北京', '上海', '广州', '深圳', '成都'], n)
education = np.random.choice(['高中', '本科', '硕士', '博士'], n)
job_type = np.random.choice(['技术', '销售', '管理', '运营'], n)

# 数值特征
age = np.random.randint(18, 60, n)
income = np.random.lognormal(8.5, 0.8, n)

# 目标变量（二分类）
# 高学历和高收入有更高的批准概率
edu_score = {'高中': 0.3, '本科': 0.5, '硕士': 0.7, '博士': 0.8}
prob = np.array([edu_score[e] for e in education]) * (income / income.max())
y = (np.random.rand(n) < prob).astype(int)

data = pd.DataFrame({
    'city': city,
    'education': education,
    'job_type': job_type,
    'age': age,
    'income': income,
})

print(f"数据形状: {data.shape}")
print(f"正样本比例: {y.mean():.2%}")
data.head()

数据形状: (2000, 5)
正样本比例: 7.15%


,city,education,job_type,age,income
0,深圳,博士,技术,33,1349.231662
1,成都,硕士,管理,26,6246.134993
2,广州,博士,技术,29,10362.490646
3,成都,硕士,技术,55,3526.001431
4,成都,本科,销售,36,8608.974544


## 2. WOEEncoder（证据权重编码）

WOE 编码是评分卡建模中最常用的编码方式，具有以下特点：
- 适用于二分类问题
- 直接对类别特征计算WOE
- 提供 IV 值评估特征预测能力
- 输出值具有可解释性

In [3]:
# WOE 编码
woe_encoder = WOEEncoder(
    cols=['city', 'education', 'job_type'],
)

X_woe = woe_encoder.fit_transform(data, y)

print("WOE 编码后数据:")
print(X_woe[['city', 'education', 'job_type']].head())
print()
print("IV值（特征预测能力）:")
print(woe_encoder.summary())

WOE 编码后数据:
       city  education  job_type
0 -0.079686  -0.328028 -0.163792
1 -0.009458  -0.114303  0.105165
2 -0.054725  -0.328028 -0.163792
3 -0.009458  -0.114303 -0.163792
4 -0.009458  -0.018687 -0.259648

IV值（特征预测能力）:
          特征     IV值   预测能力
1  education  0.1074  中等预测力
2   job_type  0.0565   弱预测力
0       city  0.0057   无预测力


## 3. TargetEncoder（目标编码）

用目标变量的均值对每个类别进行编码：
- 分类任务：该类别的正样本比例
- 回归任务：该类别的目标变量均值
- 支持平滑处理防止过拟合

In [4]:
# Target 编码
target_encoder = TargetEncoder(
    cols=['city', 'education'],
    smoothing=1.0,  # 平滑参数
)

X_target = target_encoder.fit_transform(data, y)

print("Target 编码后数据:")
print(X_target[['city', 'education']].head())
print()
print("编码映射（education）:")
print(target_encoder.mapping_['education'])

Target 编码后数据:
       city  education
0  0.075747   0.096107
1  0.070906   0.078660
2  0.073872   0.096107
3  0.070906   0.078660
4  0.070906   0.071868

编码映射（education）:
{'博士': 0.09610652591170825, '本科': 0.07186782786885246, '硕士': 0.07866012396694215, '高中': 0.039278864970645797, nan: 0.0715, '__UNKNOWN__': 0.0715}


## 4. CountEncoder（计数编码）

用每个类别的出现次数或频率进行编码：
- 适用于高基数类别特征
- 捕捉类别的流行度信息
- 可以将低频类别合并

In [5]:
# Count 编码
count_encoder = CountEncoder(
    cols=['city', 'education'],
    normalize=True,  # 返回频率而不是计数
)

X_count = count_encoder.fit_transform(data)

print("Count 编码后数据（频率）:")
print(X_count[['city', 'education']].head())
print()
print("各类别频率:")
print(count_encoder.mapping_['education'])

Count 编码后数据（频率）:
     city  education
0  0.1980     0.2600
1  0.2045     0.2415
2  0.1895     0.2600
3  0.2045     0.2415
4  0.2045     0.2435

各类别频率:
{'博士': 0.26, '高中': 0.255, '本科': 0.2435, '硕士': 0.2415, nan: 0.0, '__UNKNOWN__': 0.0}


## 5. OneHotEncoder（独热编码）

将每个类别转换为一个二进制列：
- 适用于类别数量不多的特征
- 线性模型和神经网络的常用编码方式
- 支持删除一列避免多重共线性

In [6]:
# OneHot 编码
ohe_encoder = OneHotEncoder(
    cols=['education'],
    drop='first',  # 删除第一列避免共线性
)

X_ohe = ohe_encoder.fit_transform(data)

print(f"独热编码前形状: {data.shape}")
print(f"独热编码后形状: {X_ohe.shape}")
print()
print("新增列:")
edu_cols = [c for c in X_ohe.columns if 'education_' in c]
print(edu_cols)

独热编码前形状: (2000, 5)
独热编码后形状: (2000, 7)

新增列:
['education_本科', 'education_硕士', 'education_高中']


## 6. OrdinalEncoder（序数编码）

将类别映射为整数：
- 适用于树模型
- 支持自定义映射
- 保留单一特征维度

In [7]:
# Ordinal 编码
ordinal_encoder = OrdinalEncoder(
    cols=['education'],
)

X_ordinal = ordinal_encoder.fit_transform(data)

print("Ordinal 编码映射:")
print(ordinal_encoder.get_mapping('education'))
print()
print("编码后数据:")
print(X_ordinal['education'].head(10))

Ordinal 编码映射:
{'博士': 0, '本科': 1, '硕士': 2, '高中': 3, nan: -1, '__UNKNOWN__': -1}

编码后数据:
0    0
1    2
2    0
3    2
4    1
5    3
6    1
7    3
8    3
9    3
Name: education, dtype: int64


## 7. 回归任务编码器演示

以下编码器特别适用于回归任务：

In [8]:
# 创建回归目标变量
y_reg = income + np.random.randn(n) * 1000

# QuantileEncoder - 使用分位数编码
quantile_encoder = QuantileEncoder(
    cols=['education'],
    quantile=0.5,  # 中位数
)
X_quantile = quantile_encoder.fit_transform(data, y_reg)
print("QuantileEncoder 编码结果:")
print(X_quantile[['education']].head())

QuantileEncoder 编码结果:
     education
0  4601.192728
1  4962.109988
2  4601.192728
3  4962.109988
4  4783.859722


In [9]:
# CatBoostEncoder - CatBoost风格的编码
catboost_encoder = CatBoostEncoder(
    cols=['city'],
    random_state=42,
)
X_catboost = catboost_encoder.fit_transform(data, y_reg)
print("CatBoostEncoder 编码结果:")
print(X_catboost[['city']].head())

CatBoostEncoder 编码结果:
          city
0  7178.527143
1  6672.448128
2  7365.703887
3  6686.799091
4  6823.989455


## 8. 编码器比较

比较不同编码器对同一特征的编码结果：

In [10]:
# 比较不同编码器
comparison = pd.DataFrame({
    '原始值': data['education'],
    'WOE': WOEEncoder(cols=['education']).fit_transform(data, y)['education'],
    'Target': TargetEncoder(cols=['education']).fit_transform(data, y)['education'],
    'Count': CountEncoder(cols=['education'], normalize=True).fit_transform(data)['education'],
    'Ordinal': OrdinalEncoder(cols=['education']).fit_transform(data)['education'],
})

print("编码器比较（前10行）:")
print(comparison.head(10))
print()

# 查看各类别的编码统计
print("各类别 WOE 值:")
woe_enc = WOEEncoder(cols=['education']).fit(data, y)
for edu, woe in woe_enc.mapping_['education'].items():
    print(f"  {edu}: {woe:.4f}")

编码器比较（前10行）:
  原始值       WOE    Target   Count  Ordinal
0  博士 -0.328028  0.096107  0.2600        0
1  硕士 -0.114303  0.078660  0.2415        2
2  博士 -0.328028  0.096107  0.2600        0
3  硕士 -0.114303  0.078660  0.2415        2
4  本科 -0.018687  0.071868  0.2435        1
5  高中  0.600861  0.039279  0.2550        3
6  本科 -0.018687  0.071868  0.2435        1
7  高中  0.600861  0.039279  0.2550        3
8  高中  0.600861  0.039279  0.2550        3
9  高中  0.600861  0.039279  0.2550        3

各类别 WOE 值:
  博士: -0.3280
  硕士: -0.1143
  本科: -0.0187
  高中: 0.6009
  nan: 0.0000
  __UNKNOWN__: 0.0000


## 9. 使用建议

| 编码器 | 适用场景 | 优点 | 缺点 |
|--------|----------|------|------|
| **WOEEncoder** | 评分卡、二分类 | 可解释性强、有IV评估 | 只适用于二分类 |
| **TargetEncoder** | 高基数类别、树模型 | 效果好，平滑处理 | 可能过拟合 |
| **CountEncoder** | 高基数类别 | 简单、捕捉流行度 | 信息量有限 |
| **OneHotEncoder** | 低基数类别、线性模型 | 无偏、易理解 | 维度爆炸 |
| **OrdinalEncoder** | 树模型 | 保持单一维度 | 假顺序关系 |
| **QuantileEncoder** | 回归任务 | 对异常值鲁棒 | 计算复杂 |
| **CatBoostEncoder** | 高基数类别 | 防止过拟合 | 随机性 |

## 10. 导出和导入编码器

编码器可以导出映射，用于部署时复用：

In [11]:
# 拟合编码器
encoder = WOEEncoder(cols=['city', 'education'])
encoder.fit(data, y)

# 导出映射
exported = encoder.export_mapping()
print("导出的映射信息:")
print(f"编码器类型: {exported['encoder_type']}")
print(f"编码列: {exported['cols_']}")

# 在新编码器中导入映射
new_encoder = WOEEncoder()
new_encoder.import_mapping(exported)

# 使用导入的映射进行转换
X_new = new_encoder.transform(data)
print("\n使用导入映射转换成功")

导出的映射信息:
编码器类型: WOEEncoder
编码列: ['city', 'education']

使用导入映射转换成功
